In [1]:
import json
import random
import shutil
from pathlib import Path
from collections import defaultdict, Counter
from tqdm import tqdm
import numpy as np

random.seed(42)
np.random.seed(42)

# ============================================================
# PATHS
# ============================================================

COCO_IMG_DIR  = Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017")
COCO_ANN_FILE = Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/instances_train2017.json")

UA_IMG_ROOT   = Path("/kaggle/input/datasets/dimarmalade/ua-detrac-reannotated-classes/UA-DETRAC_UPD_ANN/images/train")
UA_LABEL_ROOT = Path("/kaggle/input/datasets/dimarmalade/ua-detrac-reannotated-classes/UA-DETRAC_UPD_ANN/labels/train")

CUSTOM_IMG_TRAIN = Path("/kaggle/input/datasets/trnthiidng/mix-dataset-traffic/mixed_yolo_dataset/images/train")
CUSTOM_LBL_TRAIN = Path("/kaggle/input/datasets/trnthiidng/mix-dataset-traffic/mixed_yolo_dataset/labels/train")

CUSTOM_IMG_VAL = Path("/kaggle/input/datasets/trnthiidng/mix-dataset-traffic/mixed_yolo_dataset/images/val")
CUSTOM_LBL_VAL = Path("/kaggle/input/datasets/trnthiidng/mix-dataset-traffic/mixed_yolo_dataset/labels/val")

DATA_ROOT = Path("/kaggle/working/dataset")

OUT_TRAIN_IMG = DATA_ROOT / "images/train"
OUT_TRAIN_LBL = DATA_ROOT / "labels/train"
OUT_VAL_IMG   = DATA_ROOT / "images/val"
OUT_VAL_LBL   = DATA_ROOT / "labels/val"

for d in [OUT_TRAIN_IMG, OUT_TRAIN_LBL, OUT_VAL_IMG, OUT_VAL_LBL]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# CONFIG
# ============================================================

TARGET_CLASSES = ['car','truck','bus','motor','bicycle','person']
CLASS_TO_ID = {c:i for i,c in enumerate(TARGET_CLASSES)}

CUSTOM_ID_MAP = {
    0: "bicycle",
    1: "bus",
    2: "car",
    3: "motor",
    4: "person",
    5: "truck"
}

VAL_RATIO = 0.1
N_UA = 10000
N_COCO = 5000

RARE_CLASSES = {"truck","bus","motor","bicycle"}

CLASS_WEIGHT = {
    "motor":8,"bicycle":8,"bus":6,"truck":6,
    "car":-2,"person":-2
}

MAX_CAR = 15

UA_MAP = {0:'bicycle',1:'motor',2:'car',3:'truck',4:'bus',5:'truck'}

COCO_TO_TARGET = {
    'car':'car','truck':'truck','bus':'bus',
    'motorcycle':'motor','bicycle':'bicycle','person':'person'
}

# ============================================================
# SCORE
# ============================================================

def score(cnt):
    return sum(n * CLASS_WEIGHT.get(cls,1) for cls,n in cnt.items())

# ============================================================
# SCAN UA (PRIORITY MOTOR/BICYCLE)
# ============================================================

print("Scanning UA (priority motor/bicycle)...")

ua_high, ua_low = [], []

for fpath in tqdm(UA_LABEL_ROOT.glob("*.txt")):
    cnt = Counter()

    for ln in open(fpath):
        p = ln.split()
        if len(p)==5:
            cid=int(p[0])
            if cid in UA_MAP:
                cnt[UA_MAP[cid]] += 1

    if not cnt:
        continue

    if cnt.get("car",0) > MAX_CAR:
        continue

    item = {"src":"ua","file":fpath,"cnt":cnt}

    if cnt.get("motor",0) > 0 or cnt.get("bicycle",0) > 0:
        ua_high.append(item)
    else:
        ua_low.append(item)

ua_high = sorted(ua_high, key=lambda x: score(x["cnt"]), reverse=True)
ua_low  = sorted(ua_low,  key=lambda x: score(x["cnt"]), reverse=True)

ua_selected = ua_high[:N_UA]

if len(ua_selected) < N_UA:
    ua_selected += ua_low[:(N_UA - len(ua_selected))]

print(f"UA selected: {len(ua_selected)}")

# DEBUG
ua_motor = sum(1 for x in ua_selected if x["cnt"].get("motor",0)>0)
ua_bicycle = sum(1 for x in ua_selected if x["cnt"].get("bicycle",0)>0)

print("[UA DEBUG]")
print("motor images:", ua_motor)
print("bicycle images:", ua_bicycle)

# ============================================================
# SCAN COCO
# ============================================================

print("Scanning COCO...")

coco = json.load(open(COCO_ANN_FILE))
cat_id_to_name = {c["id"]:c["name"] for c in coco["categories"]}
img_map = {img["id"]:img for img in coco["images"]}

ann_by_img = defaultdict(list)
for ann in coco["annotations"]:
    if ann.get("iscrowd",0): continue
    name = cat_id_to_name[ann["category_id"]]
    if name in COCO_TO_TARGET:
        ann_by_img[ann["image_id"]].append(ann)

coco_pool = []
for img_id, anns in ann_by_img.items():
    cnt = Counter()
    for ann in anns:
        cls = COCO_TO_TARGET[cat_id_to_name[ann["category_id"]]]
        cnt[cls]+=1
    coco_pool.append({"src":"coco","id":img_id,"cnt":cnt,"anns":anns})

coco_pool = sorted(coco_pool, key=lambda x: score(x["cnt"]), reverse=True)

# ============================================================
# SCAN CUSTOM (FILTER)
# ============================================================

print("Scanning CUSTOM...")

TARGET_CUSTOM = {"bicycle","truck","bus"}

def scan_custom(img_dir, lbl_dir):
    pool=[]
    for txt in tqdm(lbl_dir.glob("*.txt")):
        cnt = Counter()
        new_lines=[]
        has_target=False

        for ln in open(txt):
            p=ln.split()
            if len(p)!=5: continue

            old_id=int(p[0])
            if old_id not in CUSTOM_ID_MAP:
                continue

            cls=CUSTOM_ID_MAP[old_id]
            cnt[cls]+=1

            if cls in TARGET_CUSTOM:
                has_target=True

            new_lines.append((cls,p[1:]))

        if not has_target:
            continue

        if cnt.get("motor",0) > 25:
            continue

        img = img_dir/(txt.stem+".jpg")
        if not img.exists():
            img = img_dir/(txt.stem+".png")
        if not img.exists():
            continue

        pool.append({
            "src":"custom",
            "img":img,
            "cnt":cnt,
            "lines":new_lines
        })

    return pool

custom_train_pool = scan_custom(CUSTOM_IMG_TRAIN, CUSTOM_LBL_TRAIN)
custom_val_pool   = scan_custom(CUSTOM_IMG_VAL, CUSTOM_LBL_VAL)

# ============================================================
# SELECT
# ============================================================

coco_selected = coco_pool[:N_COCO]
custom_selected = custom_train_pool

all_selected = ua_selected + coco_selected + custom_selected
random.shuffle(all_selected)

n_val = int(len(all_selected)*VAL_RATIO)

val_set = all_selected[:n_val] + custom_val_pool
trn_set = all_selected[n_val:]

# ============================================================
# WRITERS
# ============================================================

def write_ua(item, img_out, lbl_out):
    stem=item["file"].stem
    img=UA_IMG_ROOT/(stem+".jpg")
    if not img.exists():
        img=UA_IMG_ROOT/(stem+".png")
    if not img.exists(): return

    lines=[]
    for ln in open(item["file"]):
        p=ln.split()
        if len(p)!=5: continue
        cid=int(p[0])
        if cid not in UA_MAP: continue
        cls=UA_MAP[cid]
        lines.append(f"{CLASS_TO_ID[cls]} {' '.join(p[1:])}")

    out=f"ua_{stem}"
    shutil.copy(img, img_out/(out+img.suffix))
    (lbl_out/(out+".txt")).write_text("\n".join(lines))


def write_coco(item, img_out, lbl_out):
    info=img_map[item["id"]]
    img_path=COCO_IMG_DIR/info["file_name"]
    if not img_path.exists(): return

    W,H=info["width"],info["height"]

    lines=[]
    for ann in item["anns"]:
        cls = COCO_TO_TARGET[cat_id_to_name[ann["category_id"]]]
        x,y,w,h=ann["bbox"]
        cx=(x+w/2)/W; cy=(y+h/2)/H
        w/=W; h/=H
        lines.append(f"{CLASS_TO_ID[cls]} {cx} {cy} {w} {h}")

    out=f"coco_{img_path.stem}"
    shutil.copy(img_path, img_out/(out+img_path.suffix))
    (lbl_out/(out+".txt")).write_text("\n".join(lines))


def write_custom(item, img_out, lbl_out):
    out=f"custom_{item['img'].stem}"

    lines=[f"{CLASS_TO_ID[cls]} {' '.join(coords)}" for cls,coords in item["lines"]]

    shutil.copy(item["img"], img_out/(out+item["img"].suffix))
    (lbl_out/(out+".txt")).write_text("\n".join(lines))


WRITERS={"ua":write_ua,"coco":write_coco,"custom":write_custom}

def process(items, img_out, lbl_out):
    for it in tqdm(items):
        WRITERS[it["src"]](it, img_out, lbl_out)

# ============================================================
# RUN
# ============================================================

print("Writing train...")
process(trn_set, OUT_TRAIN_IMG, OUT_TRAIN_LBL)

print("Writing val...")
process(val_set, OUT_VAL_IMG, OUT_VAL_LBL)

# ============================================================
# STATS
# ============================================================

print("\n=========== DATASET STATS ===========")

def count_labels(label_dir):
    dist=Counter(); total=0
    for txt in label_dir.glob("*.txt"):
        for ln in open(txt):
            p=ln.split()
            if len(p)!=5: continue
            cid=int(p[0])
            if cid<len(TARGET_CLASSES):
                dist[TARGET_CLASSES[cid]]+=1
                total+=1
    return dist,total

train_dist,train_total = count_labels(OUT_TRAIN_LBL)
val_dist,val_total     = count_labels(OUT_VAL_LBL)

final_dist = train_dist + val_dist
total_bbox = train_total + val_total

print(f"\nTotal bbox: {total_bbox}\n")

for cls in TARGET_CLASSES:
    n=final_dist[cls]
    pct=n/total_bbox*100 if total_bbox else 0
    print(f"{cls:10s}: {n:8d} ({pct:.2f}%)")

# ============================================================
# SOURCE CONTRIBUTION
# ============================================================

print("\n=========== SOURCE CONTRIBUTION ===========\n")

def count_source(label_dir, prefix):
    dist=Counter()
    for txt in label_dir.glob(f"{prefix}_*.txt"):
        for ln in open(txt):
            p=ln.split()
            if len(p)!=5: continue
            cid=int(p[0])
            if cid<len(TARGET_CLASSES):
                dist[TARGET_CLASSES[cid]]+=1
    return dist

ua = count_source(OUT_TRAIN_LBL,"ua") + count_source(OUT_VAL_LBL,"ua")
coco = count_source(OUT_TRAIN_LBL,"coco") + count_source(OUT_VAL_LBL,"coco")
custom = count_source(OUT_TRAIN_LBL,"custom") + count_source(OUT_VAL_LBL,"custom")

for cls in TARGET_CLASSES:
    u,c,m = ua[cls],coco[cls],custom[cls]
    t=u+c+m
    print(f"{cls:10s}: UA={u}, COCO={c}, CUSTOM={m}, TOTAL={t}")
# ============================================================
# WRITE YAML (YOLO)
# ============================================================

print("\nWriting data.yaml...")

yaml_content = f"""
path: {DATA_ROOT}
train: images/train
val: images/val

nc: {len(TARGET_CLASSES)}
names: {TARGET_CLASSES}
"""

(DATA_ROOT / "data.yaml").write_text(yaml_content)

print("Saved:", DATA_ROOT / "data.yaml")

# ============================================================
# SPLIT STATS (TRAIN / VAL RIÊNG)
# ============================================================

print("\n=========== SPLIT STATS ===========\n")

def count_split(label_dir):
    dist = Counter()
    total = 0

    for txt in label_dir.glob("*.txt"):
        for ln in open(txt):
            p = ln.split()
            if len(p) != 5:
                continue
            cid = int(p[0])
            if cid < len(TARGET_CLASSES):
                cls = TARGET_CLASSES[cid]
                dist[cls] += 1
                total += 1

    return dist, total

train_dist, train_total = count_split(OUT_TRAIN_LBL)
val_dist, val_total     = count_split(OUT_VAL_LBL)

print("TRAIN:")
print("  Images:", len(list(OUT_TRAIN_IMG.glob("*"))))
print("  BBox  :", train_total)
for cls in TARGET_CLASSES:
    print(f"   - {cls:10s}: {train_dist[cls]}")

print("\nVAL:")
print("  Images:", len(list(OUT_VAL_IMG.glob("*"))))
print("  BBox  :", val_total)
for cls in TARGET_CLASSES:
    print(f"   - {cls:10s}: {val_dist[cls]}")

# ============================================================
# GLOBAL CHECK (IMPORTANT)
# ============================================================

print("\n=========== SANITY CHECK ===========\n")

total_imgs = len(list(OUT_TRAIN_IMG.glob("*"))) + len(list(OUT_VAL_IMG.glob("*")))
total_lbls = len(list(OUT_TRAIN_LBL.glob("*.txt"))) + len(list(OUT_VAL_LBL.glob("*.txt")))

print("Total images:", total_imgs)
print("Total labels:", total_lbls)

if total_imgs != total_lbls:
    print(" WARNING: image-label mismatch!")
else:
    print("Image-label OK")
# ============================================================
# STEP 9 — Train YOLOv11
# ============================================================
!pip install -q ultralytics
import zipfile
from ultralytics import YOLO
from shutil import copy2

RUN_NAME    = "yolo11m_traffic"
RUNS_DIR    = DATA_ROOT / "runs"
CKPT_PATH   = RUNS_DIR / RUN_NAME / "weights" / "last.pt"
KAGGLE_CKPT = Path("/kaggle/input/models/trnthiidng/yolov11m-newcheckpoint/other/default/5/last (7).pt")

if CKPT_PATH.exists():
    print("\nResume from working checkpoint:", CKPT_PATH)
    model  = YOLO(str(CKPT_PATH))
    resume = True

elif KAGGLE_CKPT.exists():
    print("\nResume from Kaggle checkpoint:", KAGGLE_CKPT)
    CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
    copy2(KAGGLE_CKPT, CKPT_PATH)
    model  = YOLO(str(CKPT_PATH))
    resume = True

else:
    print("\nStart fresh from pretrained yolo11m.pt")
    model  = YOLO("yolo11m.pt")
    resume = False

try:
    model.train(
        data      = str(DATA_ROOT / "data.yaml"),
        imgsz     = 640,
        epochs    = 150,
        batch     = -1,
        workers   = 2,
        project   = str(RUNS_DIR),
        name      = RUN_NAME,
        resume    = resume,
        # Augmentation
        hsv_h      = 0.015,
        hsv_s      = 0.7,
        hsv_v      = 0.4,
        degrees    = 0.0,
        translate  = 0.1,
        scale      = 0.5,
        fliplr     = 0.5,
        flipud     = 0.0,
        shear      = 0.0,
        mosaic     = 1.0,
        mixup      = 0.0,
        copy_paste = 0.1,
    )

finally:
    wdir = RUNS_DIR / RUN_NAME / "weights"
    if wdir.exists():
        zip_path = DATA_ROOT / "weights_checkpoint.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in wdir.glob("*.pt"):
                zf.write(f, arcname=f.name)
        print(f"\nWeights zipped → {zip_path}")

Scanning UA (priority motor/bicycle)...


83756it [12:59, 107.46it/s]


UA selected: 10000
[UA DEBUG]
motor images: 5982
bicycle images: 2900
Scanning COCO...
Scanning CUSTOM...


13364it [02:24, 92.70it/s]
2930it [00:30, 96.84it/s]


Writing train...


100%|██████████| 21096/21096 [04:07<00:00, 85.08it/s]


Writing val...


100%|██████████| 4109/4109 [00:50<00:00, 80.64it/s]



=========== DATASET STATS ===========

Total bbox: 221109

car       :    94848 (42.90%)
truck     :    17390 (7.86%)
bus       :    14908 (6.74%)
motor     :    41372 (18.71%)
bicycle   :    15932 (7.21%)
person    :    36659 (16.58%)

=========== SOURCE CONTRIBUTION ===========

car       : UA=40575, COCO=4986, CUSTOM=49287, TOTAL=94848
truck     : UA=6372, COCO=4446, CUSTOM=6572, TOTAL=17390
bus       : UA=6908, COCO=2626, CUSTOM=5374, TOTAL=14908
motor     : UA=6585, COCO=6729, CUSTOM=28058, TOTAL=41372
bicycle   : UA=3660, COCO=4997, CUSTOM=7275, TOTAL=15932
person    : UA=0, COCO=13320, CUSTOM=23339, TOTAL=36659

Writing data.yaml...
Saved: /kaggle/working/dataset/data.yaml

=========== SPLIT STATS ===========

TRAIN:
  Images: 21096
  BBox  : 177316
   - car       : 76277
   - truck     : 14518
   - bus       : 12396
   - motor     : 31976
   - bicycle   : 13379
   - person    : 28770

VAL:
  Images: 4109
  BBox  : 43793
   - car       : 18571
   - truck     : 2872
   - bus    